# Esercizio 17 – Mini-melodia "Scala e salto"

Creazione di una partitura con **music21**: scala ascendente, salti e risoluzione in Do maggiore.

- **Tonalità:** Do maggiore
- **Tempo:** 100 BPM
- **Misura:** 4/4
- **Misure:** 4

In [1]:
# Installazione delle dipendenze
!pip install music21 -q
!apt-get install -y fluidsynth > /dev/null 2>&1
!pip install midi2audio -q

In [2]:
from music21 import stream, note, meter, tempo, instrument, key
from midi2audio import FluidSynth
from IPython.display import Audio, display, Image
import os

## 1. Creazione della partitura

In [3]:
# ── Definizione delle 4 misure ──
# Formato: (nome_nota, quarterLength)
misure = [
    # Misura 1: scala ascendente C-D-E-F (quarti)
    [('C4', 1), ('D4', 1), ('E4', 1), ('F4', 1)],
    # Misura 2: G mezza, E e D quarti
    [('G4', 2), ('E4', 1), ('D4', 1)],
    # Misura 3: salto C-E-G-C5 (quarti)
    [('C4', 1), ('E4', 1), ('G4', 1), ('C5', 1)],
    # Misura 4: discesa B-A quarti, G mezza (risoluzione)
    [('B4', 1), ('A4', 1), ('G4', 2)],
]

# ── Costruzione dello Score ──
score = stream.Score()
score.metadata = stream.Score().metadata

# Creiamo una Part con strumento Piano
piano_part = stream.Part()
piano_part.insert(0, instrument.Piano())

# Indicazione di tempo e tonalità (inserite nella prima misura)
ts = meter.TimeSignature('4/4')
bpm = tempo.MetronomeMark(number=100)
ks = key.Key('C')

# Inserimento delle misure
for i, misura_notes in enumerate(misure):
    m = stream.Measure(number=i + 1)
    if i == 0:
        m.append(ts)
        m.append(bpm)
        m.append(ks)
    for pitch, duration in misura_notes:
        n = note.Note(pitch)
        n.quarterLength = duration
        m.append(n)
    piano_part.append(m)

score.insert(0, piano_part)

print('✅ Partitura creata con successo!')
print(f'   Misure: {len(misure)}')
print(f'   Tonalità: Do maggiore')
print(f'   Tempo: 100 BPM')
print(f'   Strumento: Piano')

✅ Partitura creata con successo!
   Misure: 4
   Tonalità: Do maggiore
   Tempo: 100 BPM
   Strumento: Piano


## 2. Visualizzazione dello spartito

In [4]:
# Visualizzazione come testo
print('─' * 50)
print('PARTITURA – Mini-melodia "Scala e salto"')
print('─' * 50)
for el in piano_part.recurse():
    if isinstance(el, note.Note):
        dur_name = {
            1.0: 'quarto ♩',
            2.0: 'metà 𝅗𝅥',
            0.5: 'ottavo ♪',
            4.0: 'intero 𝅝'
        }.get(el.quarterLength, str(el.quarterLength))
        print(f'  Mis. {el.measureNumber} │ {el.nameWithOctave:>4}  {dur_name}')
print('─' * 50)

──────────────────────────────────────────────────
PARTITURA – Mini-melodia "Scala e salto"
──────────────────────────────────────────────────
  Mis. 1 │   C4  quarto ♩
  Mis. 1 │   D4  quarto ♩
  Mis. 1 │   E4  quarto ♩
  Mis. 1 │   F4  quarto ♩
  Mis. 2 │   G4  metà 𝅗𝅥
  Mis. 2 │   E4  quarto ♩
  Mis. 2 │   D4  quarto ♩
  Mis. 3 │   C4  quarto ♩
  Mis. 3 │   E4  quarto ♩
  Mis. 3 │   G4  quarto ♩
  Mis. 3 │   C5  quarto ♩
  Mis. 4 │   B4  quarto ♩
  Mis. 4 │   A4  quarto ♩
  Mis. 4 │   G4  metà 𝅗𝅥
──────────────────────────────────────────────────


In [5]:
# Visualizzazione grafica dello spartito (immagine PNG)
# music21 usa LilyPond o MuseScore se disponibili; in Colab usiamo il metodo integrato
try:
    # Prova a generare l'immagine con music21
    img_path = score.write('musicxml.png')
    display(Image(filename=str(img_path)))
    print('✅ Spartito visualizzato come immagine')
except Exception as e:
    print(f'⚠️  Visualizzazione grafica non disponibile in questo ambiente: {e}')
    print('   Per visualizzare lo spartito grafico, installa MuseScore o LilyPond.')
    print('   La versione testuale è stata mostrata nella cella precedente.')
    # Fallback: mostra con .show('text')
    score.show('text')

⚠️  Visualizzazione grafica non disponibile in questo ambiente: Cannot find a path to the 'mscore' file at /usr/bin/mscore3 -- download MuseScore
   Per visualizzare lo spartito grafico, installa MuseScore o LilyPond.
   La versione testuale è stata mostrata nella cella precedente.
{0.0} <music21.stream.Part 0x78e8d9056840>
    {0.0} <music21.instrument.Piano 'Piano'>
    {0.0} <music21.stream.Measure 1 offset=0.0>
        {0.0} <music21.tempo.MetronomeMark Quarter=100>
        {0.0} <music21.key.Key of C major>
        {0.0} <music21.meter.TimeSignature 4/4>
        {0.0} <music21.note.Note C>
        {1.0} <music21.note.Note D>
        {2.0} <music21.note.Note E>
        {3.0} <music21.note.Note F>
    {4.0} <music21.stream.Measure 2 offset=4.0>
        {0.0} <music21.note.Note G>
        {2.0} <music21.note.Note E>
        {3.0} <music21.note.Note D>
    {8.0} <music21.stream.Measure 3 offset=8.0>
        {0.0} <music21.note.Note C>
        {1.0} <music21.note.Note E>
        {2.0} 

## 3. Esportazione MIDI

In [6]:
# Salvataggio del file MIDI
midi_path = 'esercizio_melodia.mid'
score.write('midi', fp=midi_path)
print(f'✅ File MIDI salvato: {midi_path}')
print(f'   Dimensione: {os.path.getsize(midi_path)} bytes')

✅ File MIDI salvato: esercizio_melodia.mid
   Dimensione: 208 bytes


## 4. Riproduzione audio in Colab

In [7]:
# Conversione MIDI → WAV con FluidSynth per l'ascolto in Colab
wav_path = 'esercizio_melodia.wav'

# Cerca il soundfont disponibile nel sistema
soundfont_paths = [
    '/usr/share/sounds/sf2/FluidR3_GM.sf2',
    '/usr/share/sounds/sf2/default-GM.sf2',
    '/usr/share/soundfonts/FluidR3_GM.sf2',
]

sf_path = None
for p in soundfont_paths:
    if os.path.exists(p):
        sf_path = p
        break

if sf_path is None:
    # Scarica un soundfont se non presente
    print('⏳ Scaricamento soundfont...')
    !apt-get install -y fluid-soundfont-gm > /dev/null 2>&1
    for p in soundfont_paths:
        if os.path.exists(p):
            sf_path = p
            break

if sf_path:
    fs = FluidSynth(sf_path)
    fs.midi_to_audio(midi_path, wav_path)
    print(f'✅ File WAV generato: {wav_path}')
    print(f'\n🎵 Ascolta la melodia:')
    display(Audio(wav_path))
else:
    print('❌ Soundfont non trovato. Prova: !apt-get install fluid-soundfont-gm')
    print('   Il file MIDI è comunque disponibile per il download.')

✅ File WAV generato: esercizio_melodia.wav

🎵 Ascolta la melodia:


## 5. Analisi rapida della melodia

In [8]:
# Riepilogo analitico
all_notes = [n for n in piano_part.recurse().notes]
pitches = [n.nameWithOctave for n in all_notes]
intervals = []
for i in range(len(all_notes) - 1):
    from music21 import interval
    intv = interval.Interval(all_notes[i], all_notes[i+1])
    intervals.append(intv.name)

print('📊 Analisi della melodia')
print(f'   Note totali: {len(all_notes)}')
print(f'   Ambitus: {all_notes[0].pitch.nameWithOctave} → C5')
print(f'   Sequenza: {" → ".join(pitches)}')
print(f'   Intervalli: {" | ".join(intervals)}')
print(f'\n   Mis.1: Scala ascendente (C→F) – moto congiunto')
print(f'   Mis.2: Salto a G, poi discesa – arco melodico')
print(f'   Mis.3: Arpeggio di Do maggiore (C-E-G-C) – salto')
print(f'   Mis.4: Discesa per grado verso G – semicadenza')

📊 Analisi della melodia
   Note totali: 14
   Ambitus: C4 → C5
   Sequenza: C4 → D4 → E4 → F4 → G4 → E4 → D4 → C4 → E4 → G4 → C5 → B4 → A4 → G4
   Intervalli: M2 | M2 | m2 | M2 | m3 | M2 | M2 | M3 | m3 | P4 | m2 | M2 | M2

   Mis.1: Scala ascendente (C→F) – moto congiunto
   Mis.2: Salto a G, poi discesa – arco melodico
   Mis.3: Arpeggio di Do maggiore (C-E-G-C) – salto
   Mis.4: Discesa per grado verso G – semicadenza


---
**Fine Esercizio 17** – Partitura creata, visualizzata, esportata in MIDI e riprodotta in audio. 🎶